# Experiments on the Quantinuum emulator `reimei-E`

This notebook runs the experiments of the paper that use the emulator through Quantinuum Nexus:

1. the noise strength inferred from echo circuits,
2. the two-copy measurement of a block of the Bures metric,
3. the federated training with IID data and with a Dirichlet label partition.

**Run records.** Every cell that executes circuits ends by showing a *run record*: which program
produced the result (repository, commit, state of the working tree, entry point), when (start and end
of the cell, submission and completion of every job), and on which device (backend, device name,
error-rate multipliers, job identifiers, shots, cost).  The record is also saved as JSON next to the
results.  The commit is logged at the moment a job is submitted.

**Cache.** The shot tables of finished jobs are stored in the `results/` folders of this repository.
A cell whose results are all in the cache submits nothing and needs no login; its record then lists the
jobs that produced the cached results, with their original submission times and commits.  To repeat a
measurement on the emulator, remove its cache folder, log in with `qnx login`, and run the cell again.

In [ ]:
import os, pathlib
ROOT = pathlib.Path.cwd() if (pathlib.Path.cwd() / "qfedqgt").is_dir() else pathlib.Path.cwd().parent
os.chdir(ROOT)                       # the scripts write to results/ relative to the repository root

from qfedqgt import provenance
info = provenance.program_info()
print("repository :", info["repository"])
print("commit     :", info["commit"])
print("modified   :", info["modified"] or "nothing (clean working tree)")

## 0. Login check (only needed before a cell that submits jobs)

The check reports whether the stored Nexus login is valid.  It does not print the account.

In [ ]:
from qfedqgt.nexus import login_status
print(login_status())

## 1. Noise strength from echo circuits

The echo circuit `U` followed by `U^dagger` returns the all-zero outcome with a probability equal to the
purity of the state for global depolarizing noise.  The first cell uses the two-layer circuit of the
simulations, the second the one-layer circuit of the emulator experiment.

In [ ]:
%run scripts/nexus_geometry.py --device reimei-E --scales 1 50 150 200 300 500 --shots 100

In [ ]:
%run scripts/nexus_geometry.py --device reimei-E --scenario emulator_iid --scales 1 100 200 300 400 500 --shots 100

## 2. Two-copy measurement of the Bures metric

A destructive SWAP test on two copies of the state gives the overlaps `Tr[rho_a rho_b]`; with the
parameter-shift rule they give `Tr[(d_i rho)(d_j rho)]`, and the purity gives the factor that turns it
into the Bures metric.  The cell measures the 8 x 8 block of the second layer at one input.

In [ ]:
%run scripts/nexus_metric.py --backend nexus --device reimei-E --scales 1 50 --shots 200 --layer 1

## 3. Federated training on the emulator

Six seeds with IID data and six seeds with a Dirichlet label partition advance round by round together,
and the circuits of a round are submitted at once.  Four clients take part: two reference clients with
30 shots per circuit and two clients with 300-fold error rates and one shot per circuit.  The step sizes
were selected with the local gate-level model on separate tuning seeds
(`scripts/run_campaign.py tune --scenarios emulator_iid emulator_dirichlet`).  After the last round the
models of the last five rounds are evaluated on the emulator with the reference error rates.

In [ ]:
%run scripts/run_emulator.py --backend nexus --device reimei-E --name emulator-reimei-E

## 4. Summary of the training

`device` is the test accuracy measured on the emulator (mean over the last five rounds), and `noiseless`
is the accuracy of the noiseless circuit with the same parameters.

In [ ]:
import json
import numpy as np

summary = json.load(open(ROOT / "results" / "emulator-reimei-E" / "summary.json"))
methods = summary["methods"]
for scenario in ("emulator_iid", "emulator_dirichlet"):
    runs = [r for r in summary["runs"] if r["scenario"] == scenario]
    dev = {m: np.array([r["final"][m]["accuracy_device"] for r in runs]) for m in methods}
    exact = {m: np.array([r["final"][m]["accuracy_noiseless"] for r in runs]) for m in methods}
    print(f"{scenario}: {len(runs)} seeds")
    for m in methods:
        se = dev[m].std(ddof=1) / np.sqrt(len(dev[m]))
        print(f"  {m:11s} device {dev[m].mean():.3f} +/- {se:.3f}   noiseless {exact[m].mean():.3f}")
    for m in methods[:-1]:
        d = 100 * (dev[methods[-1]] - dev[m])
        print(f"  {methods[-1]} - {m:11s} {d.mean():+.1f} +/- {d.std(ddof=1) / np.sqrt(len(d)):.1f} points, "
              f"higher in {(d > 0).sum()} of {len(d)} seeds")

In [ ]:
%run scripts/make_figures.py emulator

In [ ]:
# the saved run records can be shown again at any time
provenance.load_record(str(ROOT / "results" / "metric" / "reimei-E-provenance.json"))